# Prévoir la demande horaire de vélos en location avec Spark

Ce TP traite une prévision de ventes comme une régression : combien de vélos seront loués à chaque heure du lendemain ? Prévoir les ventes d'un magasin ou d'un produit, c'est le même problème, avec des tickets de caisse à la place des locations. Vous y construisez des variables de décalage et des moyennes glissantes avec les fenêtres de Spark, sans laisser passer d'information du futur, coupez les données dans le temps, mesurez une base naïve, puis entraînez un `GBTRegressor` qui doit la battre. Vous vérifiez ensuite ce qu'une découpe aléatoire et une fuite font croire, puis passez à un modèle par série avec `applyInPandas`, l'une des deux façons de prévoir par magasin ou par produit. Un dernier exercice, facultatif, règle le GBT sur une validation qui respecte l'ordre du temps.

La prévision se fait à minuit, une fois la veille close, pour chaque heure du lendemain : les locations sont connues jusqu'à la veille incluse, le calendrier l'est à l'avance, et la météo du lendemain serait une prévision météo. Ce TP prend la météo observée à sa place, ce qui rend ses erreurs un peu optimistes.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Le dernier, facultatif, ne sert à aucun autre. Spark tourne ici en mode local, sur les deux cœurs de la machine de Colab : chaque entraînement prend de 30 secondes à une minute.

## Installation et données

Les données sont celles de [Capital Bikeshare](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), le service de vélos en libre-service de Washington, en 2011 et 2012 (Fanaee-T, UCI Machine Learning Repository, licence CC BY 4.0) : 17 379 lignes, une par heure. Aucune n'a zéro location : les heures sans location n'y figurent pas, comme celles où le service était fermé. Les colonnes utiles ici : le jour (`dteday`) et l'heure (`hr`), le jour de la semaine (`weekday`, 0 pour dimanche), le jour ouvré (`workingday`), le jour férié (`holiday`), la météo (`weathersit`, de 1 pour un temps dégagé à 4 pour une forte pluie), la température (`temp`, divisée par 41 °C), l'humidité (`hum`), le vent (`windspeed`), et les locations de l'heure : celles des clients occasionnels (`casual`), des abonnés (`registered`) et leur total (`cnt`), la cible.

In [ ]:
!pip install pyspark
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-sharing/hour.csv

In [ ]:
import os
import sys
import time

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor

La session tourne en local sur deux cœurs, avec quatre partitions pour les agrégations au lieu des 200 par défaut. Les barres de progression de Spark sont masquées : elles encombreraient les sorties. Les processus Python que Spark lance pour exécuter la fonction pandas du dernier exercice prennent l'interpréteur de ce notebook, `sys.executable` : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut être d'une autre version de Python que celui du driver, ce que Spark refuse.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("velos-prevision")
  .master("local[2]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

`velos` est la table entière. La colonne `jour` numérote les jours depuis le 1er janvier 2011 : les fenêtres des exercices s'ordonnent sur elle. `CALENDRIER` liste les variables connues au moment de prévoir : le calendrier, et la météo, qui tient lieu de prévision météo. La moyenne des locations par heure montre que le service grandit d'une année à l'autre.

In [ ]:
velos = (
  spark.read.csv("hour.csv", header=True, inferSchema=True)
  .withColumn("jour", F.datediff("dteday", F.lit("2011-01-01")))
  .cache()
)
CALENDRIER = [
  "hr",
  "weekday",
  "workingday",
  "holiday",
  "weathersit",
  "temp",
  "hum",
  "windspeed",
]
print(velos.count())
velos.groupBy("yr").agg(F.round(F.avg("cnt"), 1).alias("moyenne")).show()

## 1. Décaler la série

La base naïve et les premières variables du modèle sont des valeurs passées de la série : les locations de la même heure la veille, ou la semaine précédente. Ajoutez à `velos`, dans `essai`, deux colonnes calculées avec `F.lag` sur une fenêtre par heure de la journée (`partitionBy("hr")`), ordonnée par `jour` : `precedent`, les locations de la ligne précédente de la même heure, et `ecart`, le nombre de jours qui la séparent de la ligne courante. Combien de lignes ont un écart de plus d'un jour ?

Ajoutez ensuite `veille` et `semaine`, les locations de la même heure exactement un jour et sept jours plus tôt, avec une fenêtre bornée sur les valeurs de `jour` (`rangeBetween`) plutôt que sur un nombre de lignes. Affichez ces colonnes pour les heures de 7 h à 9 h du 31 octobre 2012, juste après le passage de l'ouragan Sandy sur Washington : d'où vient `precedent`, et que vaut `veille` ? Combien de lignes n'ont pas de `veille` ?


**Documentation utile**

- [pyspark.sql.functions.lag](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lag.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`withColumns` ajoute plusieurs colonnes d'un coup, d'un dictionnaire nom → expression. `F.lag(colonne)` rend la valeur de la ligne précédente dans l'ordre de la fenêtre, quelle que soit sa date : appliqué à `jour`, il donne le jour de cette ligne. `rangeBetween(debut, fin)` borne la fenêtre sur les valeurs de la colonne d'ordre, relativement à la ligne courante : `rangeBetween(-1, -1)` ne garde que la ligne dont `jour` vaut un de moins, s'il y en a une. Une agrégation, `F.max` par exemple, en rend alors la valeur, ou `null`. Vous devez trouver 136 lignes dont la précédente est à plus d'un jour.

```python
par_heure = Window.partitionBy(...).orderBy(...)
essai = velos.withColumns(
  {
    "precedent": F.lag(...).over(par_heure),
    "ecart": F.col("jour") - F.lag(...).over(par_heure),
    "veille": F.max(...).over(par_heure.rangeBetween(..., ...)),
    "semaine": F.max(...).over(par_heure.rangeBetween(..., ...)),
  }
)
print(essai.filter(...).count())
essai.filter("dteday = '2012-10-31' AND hr BETWEEN 7 AND 9").select(
  "dteday", "hr", "cnt", "precedent", "ecart", "veille", "semaine"
).show()
print(essai.filter(...).count())
```

### Solution

In [ ]:
par_heure = Window.partitionBy("hr").orderBy("jour")
essai = velos.withColumns(
  {
    "precedent": F.lag("cnt").over(par_heure),
    "ecart": F.col("jour") - F.lag("jour").over(par_heure),
    "veille": F.max("cnt").over(par_heure.rangeBetween(-1, -1)),
    "semaine": F.max("cnt").over(par_heure.rangeBetween(-7, -7)),
  }
)
print(essai.filter("ecart > 1").count())
essai.filter("dteday = '2012-10-31' AND hr BETWEEN 7 AND 9").select(
  "dteday", "hr", "cnt", "precedent", "ecart", "veille", "semaine"
).show()
print(essai.filter("veille IS NULL").count())
# 136 lignes ont leur ligne précédente à plus d'un jour : F.lag compte des lignes,
# pas des jours, et les heures sans location manquent. Le 31 octobre 2012 de 7 h à
# 9 h, precedent vient du dimanche 28 octobre, trois jours plus tôt : les 29 et 30,
# pendant l'ouragan Sandy, presque toutes les heures manquent. veille y est nulle,
# ce qui est juste. 160 lignes n'ont pas de veille : les 24 heures du premier jour,
# et les 136 lendemains d'une heure manquante.

## 2. Moyennes glissantes, sans fuite

Écrivez la fonction `ajouter_decalages(donnees, cles, cible)`, qui ajoute à `donnees` quatre variables calculées sur la colonne `cible` de chaque série, une série étant identifiée par les colonnes de la liste `cles` (vide ici : une seule série ; le dernier exercice en aura plusieurs) :

- `veille` et `semaine`, comme à l'exercice 1 ;
- `moy_7j`, la moyenne de la même heure sur les 7 jours précédents ;
- `moy_4s`, la moyenne de la même heure, le même jour de la semaine, sur les 4 semaines précédentes.

Aucune fenêtre n'inclut le jour à prévoir : la prévision se fait à minuit, la veille close. Appliquez-la à `velos`, avec `cnt` pour cible, dans `decales`, en cache, et affichez ces colonnes pour les heures de 7 h à 9 h du lundi 3 décembre 2012. Pourquoi `moy_4s` y est-elle plus proche de `cnt` que `moy_7j` ?

In [ ]:
def ajouter_decalages(donnees, cles, cible):
  # Votre code ici
  ...

### Conseils supplémentaires

`partitionBy` prend plusieurs colonnes : `Window.partitionBy(*cles, "hr")` fait une série par valeur des `cles` et par heure. Pour `moy_4s`, partitionnez aussi par `weekday` : les 4 semaines précédentes sont les jours `jour - 28` à `jour - 7`. Une fenêtre `rangeBetween(-7, -1)` s'arrête la veille, et compte les jours, pas les lignes : une heure absente n'y décale rien.

```python
def ajouter_decalages(donnees, cles, cible):
  par_heure = Window.partitionBy(*cles, "hr").orderBy("jour")
  par_creneau = Window.partitionBy(*cles, "hr", ...).orderBy("jour")
  return donnees.withColumns(
    {
      "veille": F.max(cible).over(par_heure.rangeBetween(..., ...)),
      "semaine": F.max(cible).over(par_heure.rangeBetween(..., ...)),
      "moy_7j": F.avg(cible).over(par_heure.rangeBetween(..., ...)),
      "moy_4s": F.avg(cible).over(par_creneau.rangeBetween(..., ...)),
    }
  )


decales = ajouter_decalages(velos, [], "cnt").cache()
decales.filter("dteday = '2012-12-03' AND hr BETWEEN 7 AND 9").select(
  "hr", "cnt", "veille", "semaine", "moy_7j", "moy_4s"
).orderBy("hr").show()
```

### Solution

In [ ]:
def ajouter_decalages(donnees, cles, cible):
  par_heure = Window.partitionBy(*cles, "hr").orderBy("jour")
  par_creneau = Window.partitionBy(*cles, "hr", "weekday").orderBy("jour")
  return donnees.withColumns(
    {
      "veille": F.max(cible).over(par_heure.rangeBetween(-1, -1)),
      "semaine": F.max(cible).over(par_heure.rangeBetween(-7, -7)),
      "moy_7j": F.avg(cible).over(par_heure.rangeBetween(-7, -1)),
      "moy_4s": F.avg(cible).over(par_creneau.rangeBetween(-28, -7)),
    }
  )


decales = ajouter_decalages(velos, [], "cnt").cache()
decales.filter("dteday = '2012-12-03' AND hr BETWEEN 7 AND 9").select(
  "hr", "cnt", "veille", "semaine", "moy_7j", "moy_4s"
).orderBy("hr").show()
# À 8 h, 731 locations : moy_4s, la moyenne des quatre lundis précédents, en
# prévoit 622, moy_7j 496 seulement. Elle mêle les jours ouvrés au week-end, qui n'a
# pas de pointe le matin. veille, un dimanche, en prévoit 93.

## 3. Découper dans le temps, une base naïve

Les quatre derniers mois, depuis le 1er septembre 2012 (`COUPURE`), servent de test. Gardez dans `entrainement` les lignes de `decales` d'avant cette date, et dans `test` les autres, sans les lignes où une variable de `DECALAGES` manque, en cache. Combien de lignes compte chacun ?

Écrivez la fonction `evaluer(predictions, colonne)`, qui affiche la RMSE et la MAE (`RegressionEvaluator`) de la colonne de prévisions `colonne` face à `cnt`. Évaluez sur `test` chacune des quatre variables de décalage, prise comme prévision : c'est une base naïve, sans modèle. Laquelle fait la meilleure ? Pourquoi `semaine`, la même heure la semaine précédente, bat-elle `veille` ?

In [ ]:
COUPURE = "2012-09-01"
DECALAGES = ["veille", "semaine", "moy_7j", "moy_4s"]


def evaluer(predictions, colonne):
  # Votre code ici
  ...

### Conseils supplémentaires

`dropna(subset=...)` ne regarde que les colonnes de la liste. Une colonne `dteday` se compare à une date écrite en texte : `F.col("dteday") < COUPURE`. Un évaluateur se règle par ses paramètres `labelCol`, `predictionCol` et `metricName` (`"rmse"`, `"mae"`), et refuse une colonne de prévisions entière, comme `veille` et `semaine` : convertissez-la d'abord en `double` (`cast`). Vous devez trouver une MAE de 67,5 locations par heure pour `semaine`.

```python
entrainement = decales.filter(...).dropna(subset=...).cache()
test = decales.filter(...).dropna(subset=...).cache()
print(entrainement.count(), test.count())


def evaluer(predictions, colonne):
  predictions = predictions.withColumn(colonne, F.col(colonne).cast(...))
  for mesure in ["rmse", "mae"]:
    evaluateur = RegressionEvaluator(labelCol=..., predictionCol=..., metricName=...)
    print(f"{colonne} {mesure} {evaluateur.evaluate(predictions):.1f}")


for colonne in DECALAGES:
  evaluer(test, colonne)
```

### Solution

In [ ]:
COUPURE = "2012-09-01"
DECALAGES = ["veille", "semaine", "moy_7j", "moy_4s"]
entrainement = (
  decales.filter(F.col("dteday") < COUPURE).dropna(subset=DECALAGES).cache()
)
test = decales.filter(F.col("dteday") >= COUPURE).dropna(subset=DECALAGES).cache()
print(entrainement.count(), test.count())


def evaluer(predictions, colonne):
  predictions = predictions.withColumn(colonne, F.col(colonne).cast("double"))
  for mesure in ["rmse", "mae"]:
    evaluateur = RegressionEvaluator(
      labelCol="cnt", predictionCol=colonne, metricName=mesure
    )
    print(f"{colonne} {mesure} {evaluateur.evaluate(predictions):.1f}")


for colonne in DECALAGES:
  evaluer(test, colonne)
# 14 129 heures d'entraînement et 2 821 de test. moy_4s fait la meilleure base (MAE
# de 58,3 locations par heure), devant semaine (67,5) : elle lisse le bruit de quatre
# semaines. veille (79,2) fait moins bien que semaine : la veille d'un lundi est un
# dimanche, quand la semaine précédente garde le même jour.

## 4. Un GBT dans un pipeline

Écrivez la fonction `entrainer(colonnes, donnees)`, qui rend un pipeline entraîné sur `donnees` : un `VectorAssembler` des `colonnes` dans `features`, puis un `GBTRegressor` (étiquette `cnt`, 20 itérations, profondeur 6, `seed=0`). Entraînez-le sur `entrainement` avec les seules variables de `CALENDRIER`, dans `modele_calendrier`, puis avec celles-ci et les `DECALAGES`, dans `modele`, en mesurant la durée de chaque entraînement, et évaluez les deux sur `test`. Lequel bat la base naïve, et de combien ?

Affichez les cinq variables les plus importantes de `modele` (les `featureImportances` du GBT, le dernier étage du pipeline). Pourquoi `modele_calendrier` fait-il moins bien que la base, alors qu'il connaît la météo ? Comparez la moyenne de `cnt` sur `entrainement` et sur `test`.

In [ ]:
def entrainer(colonnes, donnees):
  # Votre code ici
  ...

### Conseils supplémentaires

Les itérations et la profondeur sont les paramètres `maxIter` et `maxDepth`. Un `PipelineModel` garde ses étages entraînés dans `stages` : `modele.stages[-1]` est le GBT. `featureImportances` est un vecteur, dans l'ordre des colonnes assemblées ; `toArray()` le convertit en tableau NumPy.

```python
def entrainer(colonnes, donnees):
  return Pipeline(
    stages=[
      VectorAssembler(inputCols=..., outputCol="features"),
      GBTRegressor(labelCol=..., maxIter=..., maxDepth=..., seed=0),
    ]
  ).fit(donnees)


debut = time.time()
modele_calendrier = entrainer(..., entrainement)
print(f"{time.time() - debut:.0f} s")
evaluer(modele_calendrier.transform(test), "prediction")
debut = time.time()
modele = entrainer(... + ..., entrainement)
print(f"{time.time() - debut:.0f} s")
evaluer(modele.transform(test), "prediction")

importances = modele.stages[-1].featureImportances.toArray()
print(sorted(zip(importances.round(3), CALENDRIER + DECALAGES), reverse=True)[:5])
print(entrainement.agg(F.avg(...)).first()[0], test.agg(F.avg(...)).first()[0])
```

### Solution

In [ ]:
def entrainer(colonnes, donnees):
  return Pipeline(
    stages=[
      VectorAssembler(inputCols=colonnes, outputCol="features"),
      GBTRegressor(labelCol="cnt", maxIter=20, maxDepth=6, seed=0),
    ]
  ).fit(donnees)


debut = time.time()
modele_calendrier = entrainer(CALENDRIER, entrainement)
print(f"{time.time() - debut:.0f} s")
evaluer(modele_calendrier.transform(test), "prediction")
debut = time.time()
modele = entrainer(CALENDRIER + DECALAGES, entrainement)
print(f"{time.time() - debut:.0f} s")
evaluer(modele.transform(test), "prediction")

importances = modele.stages[-1].featureImportances.toArray()
print(sorted(zip(importances.round(3), CALENDRIER + DECALAGES), reverse=True)[:5])
print(entrainement.agg(F.avg("cnt")).first()[0], test.agg(F.avg("cnt")).first()[0])
# Sans décalage, le GBT fait une MAE de 83,4, pire que la base naïve (67,5) ; avec,
# 50,2 : un quart de moins que la base, et 14 % de moins que moy_4s, qui porte à elle
# seule 62 % de l'importance. Le service grandit : 183 locations par heure en
# moyenne à l'entraînement, 242 au test. Un arbre ne prédit que des moyennes de
# valeurs vues à l'entraînement : sans le niveau récent, que les décalages lui
# donnent, il ne suit pas la croissance.

## 5. Ce que `randomSplit` et une fuite font croire

Deux erreurs courantes donnent une prévision meilleure sur le papier qu'en production. Coupez au hasard les lignes de `decales` sans valeur manquante dans `DECALAGES`, 80 % pour l'entraînement et 20 % pour le test (`randomSplit`, `seed=0`), entraînez-y le modèle de l'exercice 4 (calendrier et décalages) et évaluez-le.

Ajoutez ensuite à `decales`, dans `avec_fuite`, la colonne `fuite`, calculée comme `moy_4s` sur quatre semaines, mais avec une fenêtre qui finit au jour courant : `rangeBetween(-21, 0)`. Découpez `avec_fuite` à la date de `COUPURE`, comme à l'exercice 3, entraînez le modèle avec cette variable en plus et évaluez-le. Comparez les deux erreurs à celle de l'exercice 4 : d'où vient chaque gain, et pourquoi disparaîtrait-il en production ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`randomSplit` rend une liste de tables, à dépaqueter. La fenêtre de `fuite` partitionne comme celle de `moy_4s`, par heure et par jour de la semaine ; sa borne 0 est le jour courant : l'heure à prévoir entre dans sa propre moyenne. Après le calcul sur `decales`, la découpe se refait comme à l'exercice 3.

```python
melange_entrainement, melange_test = decales.dropna(subset=...).randomSplit(
  [..., ...], seed=0
)
evaluer(entrainer(..., melange_entrainement).transform(melange_test), "prediction")

par_creneau = Window.partitionBy(..., ...).orderBy("jour")
avec_fuite = decales.withColumn(
  "fuite", F.avg("cnt").over(par_creneau.rangeBetween(..., ...))
)
entrainement_fuite = avec_fuite.filter(...).dropna(subset=...)
test_fuite = avec_fuite.filter(...).dropna(subset=...)
modele_fuite = entrainer(CALENDRIER + DECALAGES + [...], entrainement_fuite)
evaluer(modele_fuite.transform(test_fuite), "prediction")
```

### Solution

In [ ]:
melange_entrainement, melange_test = decales.dropna(subset=DECALAGES).randomSplit(
  [0.8, 0.2], seed=0
)
evaluer(
  entrainer(CALENDRIER + DECALAGES, melange_entrainement).transform(melange_test),
  "prediction",
)

par_creneau = Window.partitionBy("hr", "weekday").orderBy("jour")
avec_fuite = decales.withColumn(
  "fuite", F.avg("cnt").over(par_creneau.rangeBetween(-21, 0))
)
entrainement_fuite = avec_fuite.filter(F.col("dteday") < COUPURE).dropna(
  subset=DECALAGES
)
test_fuite = avec_fuite.filter(F.col("dteday") >= COUPURE).dropna(subset=DECALAGES)
modele_fuite = entrainer(CALENDRIER + DECALAGES + ["fuite"], entrainement_fuite)
evaluer(modele_fuite.transform(test_fuite), "prediction")
# Découpe au hasard : MAE de 36,6 environ, au lieu de 50,2. Chaque heure de test a
# ses voisines, la veille et le lendemain, dans l'entraînement : le modèle a vu le
# niveau du futur, ce qu'aucune prévision n'aura. Fuite : MAE de 38,7, la cible
# entre dans sa propre variable, avec un poids d'un quart. À minuit la veille, en
# production, cette valeur n'existe pas encore : les deux gains disparaissent.

## 6. Un modèle par série avec `applyInPandas`

Les clients occasionnels (`casual`) et les abonnés (`registered`) ne louent pas aux mêmes heures : ce sont deux séries, comme deux magasins ou deux produits. Pour des milliers de séries, deux façons de faire. Un modèle global apprend toutes les séries à la fois, l'identifiant de la série parmi ses variables : c'est ce qu'ont fait, avec LightGBM, les gagnants du concours M5 de prévision des ventes de Walmart (2020). Un modèle local par série s'entraîne par une fonction pandas que Spark applique à chaque groupe d'un `groupBy`, chaque série sur un exécuteur : c'est le cas des bibliothèques de séries temporelles comme statsforecast ou Prophet, et celui de cet exercice.

Passez `velos` au format long, une ligne par heure et par série, avec `unpivot` : les colonnes `serie` (`"casual"`, `"registered"` ou `"cnt"`, le total, gardé comme troisième série pour comparer) et `locations`, en gardant `dteday`, `jour` et les variables de `CALENDRIER`. Ajoutez-lui les décalages de chaque série (`ajouter_decalages`, sur `locations`), la colonne booléenne `avant_coupure` (date antérieure à `COUPURE`), et retirez les lignes où un décalage manque : c'est `series`.

La fonction `prevoir` ci-dessous entraîne un modèle scikit-learn sur le passé d'une série et prévoit son futur. Elle désactive son arrêt précoce (`early_stopping=False`) : au-delà de 10 000 lignes, le modèle mettrait de côté 10 % du passé pour décider quand s'arrêter, un tirage qui dépend de l'ordre des lignes, et Spark ne garantit pas cet ordre dans un groupe ; ses erreurs changeraient alors d'une exécution à l'autre. Appliquez-la à chaque série avec `groupBy("serie").applyInPandas`, en cache dans `previsions`, et calculez par série la MAE de `prevision` et celle de la base `semaine`. Le modèle bat-il la base sur chaque série ? Additionnez enfin, heure par heure, les prévisions de `casual` et de `registered` et évaluez la somme face à `cnt` sur `test` : fait-elle mieux que la série `cnt` prévue directement ?


**Documentation utile**

- [pyspark.sql.DataFrame.unpivot](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unpivot.html)

In [ ]:
VARIABLES = CALENDRIER + DECALAGES


def prevoir(serie):
  passe = serie[serie["avant_coupure"]]
  futur = serie[~serie["avant_coupure"]]
  modele_serie = HistGradientBoostingRegressor(early_stopping=False)
  modele_serie.fit(passe[VARIABLES], passe["locations"])
  return futur[["serie", "dteday", "hr", "locations", "semaine"]].assign(
    prevision=modele_serie.predict(futur[VARIABLES])
  )


# Votre code ici

### Conseils supplémentaires

`unpivot(ids, valeurs, nom_variable, nom_valeur)` garde les colonnes `ids` et fait une ligne par colonne de `valeurs`. `applyInPandas` prend la fonction et le schéma de la table qu'elle rend, en texte : `"serie string, dteday date, hr int, locations int, semaine int, prevision double"`. La somme des deux séries se calcule par `groupBy("dteday", "hr")` sur `previsions` filtrée, puis se joint à `test` sur ces deux colonnes. La série `registered` doit avoir une MAE d'environ 38 locations par heure.

```python
series = (
  ajouter_decalages(
    velos.unpivot(
      ["dteday", "jour"] + CALENDRIER, [..., ..., ...], "serie", "locations"
    ),
    [...],
    "locations",
  )
  .withColumn("avant_coupure", F.col("dteday") < COUPURE)
  .dropna(subset=DECALAGES)
)
debut = time.time()
previsions = series.groupBy(...).applyInPandas(prevoir, schema=...).cache()
previsions.groupBy("serie").agg(
  F.round(F.avg(F.abs(F.col("locations") - F.col(...))), 1).alias("mae"),
  F.round(F.avg(F.abs(F.col("locations") - F.col(...))), 1).alias("mae_base"),
).show()
print(f"{time.time() - debut:.0f} s")

somme = (
  previsions.filter(F.col("serie").isin(..., ...))
  .groupBy("dteday", "hr")
  .agg(F.sum(...).alias("somme"))
)
evaluer(somme.join(test, [..., ...]), "somme")
```

### Solution

In [ ]:
VARIABLES = CALENDRIER + DECALAGES
series = (
  ajouter_decalages(
    velos.unpivot(
      ["dteday", "jour"] + CALENDRIER,
      ["casual", "registered", "cnt"],
      "serie",
      "locations",
    ),
    ["serie"],
    "locations",
  )
  .withColumn("avant_coupure", F.col("dteday") < COUPURE)
  .dropna(subset=DECALAGES)
)
debut = time.time()
previsions = (
  series.groupBy("serie")
  .applyInPandas(
    prevoir,
    schema="serie string, dteday date, hr int, locations int, semaine int, "
    "prevision double",
  )
  .cache()
)
previsions.groupBy("serie").agg(
  F.round(F.avg(F.abs(F.col("locations") - F.col("prevision"))), 1).alias("mae"),
  F.round(F.avg(F.abs(F.col("locations") - F.col("semaine"))), 1).alias("mae_base"),
).show()
print(f"{time.time() - debut:.0f} s")

somme = (
  previsions.filter(F.col("serie").isin("casual", "registered"))
  .groupBy("dteday", "hr")
  .agg(F.sum("prevision").alias("somme"))
)
evaluer(somme.join(test, ["dteday", "hr"]), "somme")
# Le modèle bat la base sur chaque série : MAE de 10,9 contre 19,5 pour casual, 38,3
# contre 54,9 pour registered. La somme des deux séries (42,9) vaut la série cnt
# prévue directement (43,0) : découper ne gagne rien sur le total, mais donne une
# prévision par série, celle dont un magasin ou un produit a besoin. Le modèle de
# scikit-learn bat le GBT de l'exercice 4 (50,2), avec ses réglages par défaut (100
# itérations au lieu de 20) ; l'exercice 7 dit si les itérations suffisent à
# l'expliquer. Chaque série tient entière dans la mémoire d'un
# exécuteur, en pandas : des milliers de séries se répartissent ainsi sur le
# cluster, tant que chacune tient en mémoire.

## 7. Régler le GBT sur une validation dans le temps (facultatif)

Le GBT des exercices 4 et 5 garde ses réglages de départ, 20 itérations et une profondeur de 6, quand le modèle par série en fait 100. Des réglages se choisissent sur une validation, jamais sur `test` : choisis sur lui, ils l'auraient vu, et son erreur deviendrait optimiste. `TrainValidationSplit` et `CrossValidator` tirent leur validation au hasard, l'erreur de l'exercice 5 : découpez plutôt `entrainement` dans le temps, comme `test` l'a été, en `avant_aout`, ses lignes d'avant le 1er août 2012 (`VALIDATION`), et `aout`, celles du mois d'août.

Pour chaque combinaison de `maxIter` (20 ou 40) et de `maxDepth` (4, 6 ou 8), tirée d'une grille `ParamGridBuilder`, entraînez sur `avant_aout` le pipeline de l'exercice 4 (calendrier et décalages) et mesurez sa MAE sur `aout`. Réentraînez la meilleure combinaison sur tout `entrainement`, puis évaluez-la une seule fois sur `test`. Le gain mesuré en août tient-il sur les quatre mois de test ? Le GBT rattrape-t-il le modèle par série de l'exercice 6 ?

Cet exercice est facultatif : aucun autre ne s'en sert, et ses sept entraînements prennent de cinq à dix minutes.


**Documentation utile**

- [pyspark.ml.Pipeline.fit](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html#pyspark.ml.Pipeline.fit)

In [ ]:
VALIDATION = "2012-08-01"

# Votre code ici

### Conseils supplémentaires

`ParamGridBuilder().addGrid(...).build()` rend une liste de dictionnaires, un par combinaison, dont les clés sont les paramètres du GBT : `reglages[gbt.maxIter]` en lit un. `pipeline.fit(donnees, reglages)` entraîne le pipeline avec ces valeurs à la place des siennes, sans le modifier : un seul pipeline sert à toute la grille. `min(resultats, key=...)` trouve le couple de plus petite erreur. Vous devez trouver une MAE de 38,4 en août pour les réglages de départ (20 itérations, profondeur 6).

```python
avant_aout = entrainement.filter(...)
aout = entrainement.filter(...)
gbt = GBTRegressor(labelCol="cnt", seed=0)
pipeline = Pipeline(
  stages=[VectorAssembler(inputCols=CALENDRIER + DECALAGES, outputCol="features"), gbt]
)
grille = (
  ParamGridBuilder()
  .addGrid(gbt.maxIter, [..., ...])
  .addGrid(gbt.maxDepth, [..., ..., ...])
  .build()
)
evaluateur = RegressionEvaluator(labelCol="cnt", metricName="mae")
resultats = []
for reglages in grille:
  erreur = evaluateur.evaluate(pipeline.fit(..., reglages).transform(...))
  print(reglages[gbt.maxIter], reglages[gbt.maxDepth], round(erreur, 1))
  resultats.append((erreur, reglages))
meilleurs = min(resultats, key=lambda resultat: resultat[0])[1]
evaluer(pipeline.fit(..., meilleurs).transform(...), "prediction")
```

### Solution

In [ ]:
VALIDATION = "2012-08-01"
avant_aout = entrainement.filter(F.col("dteday") < VALIDATION)
aout = entrainement.filter(F.col("dteday") >= VALIDATION)
gbt = GBTRegressor(labelCol="cnt", seed=0)
pipeline = Pipeline(
  stages=[VectorAssembler(inputCols=CALENDRIER + DECALAGES, outputCol="features"), gbt]
)
grille = (
  ParamGridBuilder()
  .addGrid(gbt.maxIter, [20, 40])
  .addGrid(gbt.maxDepth, [4, 6, 8])
  .build()
)
evaluateur = RegressionEvaluator(labelCol="cnt", metricName="mae")
resultats = []
for reglages in grille:
  erreur = evaluateur.evaluate(pipeline.fit(avant_aout, reglages).transform(aout))
  print(reglages[gbt.maxIter], reglages[gbt.maxDepth], round(erreur, 1))
  resultats.append((erreur, reglages))
meilleurs = min(resultats, key=lambda resultat: resultat[0])[1]
evaluer(pipeline.fit(entrainement, meilleurs).transform(test), "prediction")
# En août, 40 itérations à la profondeur 6 font la meilleure MAE, 36,9 contre 38,4
# pour les réglages de départ ; à 40 itérations, des arbres plus profonds n'aident
# plus (37,3 à la profondeur 8). Un mois de validation est court : un écart de
# quelques dixièmes entre deux réglages n'y est pas sûr. Réentraîné sur tout
# entrainement, ce GBT fait une MAE de 47,4 sur le test, contre 50,2 : le gain
# tient, et même un peu plus (2,8 de moins sur le test, 1,5 en août). Il reste
# derrière le modèle par série (43,0) : les itérations n'expliquent qu'une part de
# l'écart.

## Fin

Arrêtez la session, qui libère les exécuteurs.

In [ ]:
spark.stop()